# Cell 1: Installation and Client Initialization
This cell installs the lightweight native engine and initializes a persistent client. This means it saves data directly to a local directory on your disk, functioning like a real database instead of disappearing from RAM when the script stops.

In [1]:
# !pip install chromadb

import chromadb

# Initialize a persistent local database client
# Chroma automatically creates a folder named 'chroma_storage' to save your data
chroma_client = chromadb.PersistentClient(path="./chroma_storage")

print("ChromaDB Client successfully initialized and connected to disk storage.")

ChromaDB Client successfully initialized and connected to disk storage.


# Cell 2: Creating a Collection
In Chroma, data is stored in isolated units called Collections. Think of a collection exactly like a Table in PostgreSQL. By default, if you don't provide an embedding model, Chroma automatically loads an internal, lightweight Sentence-Transformer model (all-MiniLM-L6-v2) to turn your incoming strings into vectors under the hood.

In [2]:
# Create a new table/collection or load it if it already exists
collection = chroma_client.get_or_create_collection(name="company_knowledge_base")

print(f"Collection '{collection.name}' is ready for dynamic ingestion.")

Collection 'company_knowledge_base' is ready for dynamic ingestion.


# Cell 3: Bulk Ingestion (Vector + Text + Metadata)
Look at how clean this API is compared to FAISS. You don't have to separate your raw text strings from your vector floats. You pass the text documents, unique tracking IDs, and custom JSON metadata dictionaries directly into a single method call.

In [4]:
# Raw enterprise technical documents to ingest
documents_list = [
    "The official company policy dictates that core servers must maintain a 99.99% uptime SLA.",
    "Employees can request hardware laptop upgrades every 24 months through the IT portal.",
    "Database backup engines automatically trigger full snapshot replications daily at 02:00 UTC."
]

# Unique metadata objects used for advanced hybrid filtering later
metadata_list = [
    {"department": "infrastructure", "importance": "critical"},
    {"department": "hr", "importance": "low"},
    {"department": "infrastructure", "importance": "high"}
]

# Unique string IDs for updating or deleting elements later
ids_list = ["doc_id_001", "doc_id_002", "doc_id_003"]

# Single atomic API call: Chroma automatically generates embeddings,
# maps them to the text documents, links the metadata, and persists it to disk.
collection.add(
    documents=documents_list,
    metadatas=metadata_list,
    ids=ids_list
)

print(f"Successfully ingested and indexed {collection.count()} documents.")

Successfully ingested and indexed 3 documents.


# Cell 4: Executing Semantic Search with Metadata Filtering
When you run a search query here, you do not need to convert your text question into a vector array manually using external code. You pass the raw string question, and Chroma converts it using the same underlying model, performs a rapid HNSW sweep, and extracts the text segments.

In [5]:
# Execute a semantic search query with an attached relational filter
search_results = collection.query(
    query_texts=["How often do we back up our main database?"],
    n_results=1,                                            # Top K matches to return
    where={"department": "infrastructure"}                  # Advanced Metadata Filtering!
)

# Extract and display the matched data payload
matched_text = search_results['documents'][0][0]
matched_meta = search_results['metadatas'][0][0]
matched_distance = search_results['distances'][0][0]

print("--- Query Results ---")
print(f"Matched Context: {matched_text}")
print(f"Metadata Tags:   {matched_meta}")
print(f"Cosine Distance: {matched_distance:.4f}")

--- Query Results ---
Matched Context: Database backup engines automatically trigger full snapshot replications daily at 02:00 UTC.
Metadata Tags:   {'importance': 'high', 'department': 'infrastructure'}
Cosine Distance: 1.0560
